In [1]:
import pandas as pd
from sqlalchemy import create_engine, Column, Integer, Float, String
from sqlalchemy.orm import declarative_base, sessionmaker
import os

# 1. Konfigurasi Koneksi Database (SQLite untuk saat ini, tersimpan di folder data/)
DB_PATH = '../data/recommendations.db'
engine = create_engine(f'sqlite:///{DB_PATH}', echo=False)
Base = declarative_base()

# 2. Mendefinisikan Skema Tabel Database
class PopularItem(Base):
    __tablename__ = 'popular_items'
    id = Column(Integer, primary_key=True, autoincrement=True)
    item_id = Column(Integer, unique=True, index=True) # Index agar pencarian secepat kilat
    score = Column(Float)

# Membuat tabel di dalam file database
Base.metadata.create_all(engine)

# 3. Memuat data dari CSV dan menghitung popularitas
df_train = pd.read_csv('../data/processed/train.csv')
item_pop = df_train.groupby('item_id').size().reset_index(name='interactions')
item_pop = item_pop.sort_values('interactions', ascending=False).head(100)

# 4. Menyimpan (Insert) data ke dalam Database
Session = sessionmaker(bind=engine)
session = Session()

# Bersihkan tabel lama jika ada
session.query(PopularItem).delete()

# Masukkan data baru
for rank, row in item_pop.iterrows():
    # Simulasi skor desimal menurun
    simulated_score = round(1.0 - (rank * 0.005), 3) 
    db_item = PopularItem(item_id=int(row['item_id']), score=simulated_score)
    session.add(db_item)

session.commit()
session.close()

print(f"Database berhasil dibuat di {DB_PATH}")
print("100 Item terlaris berhasil disimpan ke dalam tabel 'popular_items'!")

Database berhasil dibuat di ../data/recommendations.db
100 Item terlaris berhasil disimpan ke dalam tabel 'popular_items'!
